# 01 — Data Preprocessing
**Smart Health Risk Predictor — ML CCP**
Team: Muhammad Sheryar Tahir · Ifrah Adnan · Reeba Kamran

This notebook performs end-to-end preprocessing on three datasets
(**Heart Disease**, **Diabetes**, **Obesity**) and writes clean CSVs
to `dataset/clean/` for use by `02_model_training_evaluation.ipynb`.

### Steps covered
1. Dataset loading & exploration
2. Missing value handling
3. Duplicate removal
4. Outlier detection (IQR capping)
5. Feature engineering (BMI, age groups)
6. Label & one-hot encoding
7. Train/test split & StandardScaler
8. Persist cleaned dataset


In [ ]:
import os, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

DATA = Path('../dataset')
OUT  = DATA / 'clean'
OUT.mkdir(exist_ok=True)
DISEASES = {
    'heart':    {'file':'heart.csv',    'target':'target'},
    'diabetes': {'file':'diabetes.csv', 'target':'Outcome'},
    'obesity':  {'file':'obesity.csv',  'target':'NObeyesdad'},
}
DISEASES

## 1. Load & Explore

In [ ]:
def load(name):
    cfg = DISEASES[name]
    df = pd.read_csv(DATA / cfg['file'])
    print(f"--- {name} --- shape={df.shape}")
    print(df.head(3))
    print(df.describe(include='all').T.head())
    return df

# Run on all three (skip silently if file not present)
loaded = {}
for n in DISEASES:
    p = DATA / DISEASES[n]['file']
    if p.exists():
        loaded[n] = load(n)
    else:
        print(f"[skip] {p} not found - see dataset/README.md")

## 2. Missing Values & Duplicates

In [ ]:
def clean_missing(df):
    df = df.drop_duplicates()
    for c in df.columns:
        if df[c].dtype.kind in 'biufc':
            df[c] = df[c].fillna(df[c].median())
        else:
            df[c] = df[c].fillna(df[c].mode().iloc[0])
    return df

for n,df in loaded.items():
    before = df.isna().sum().sum()
    loaded[n] = clean_missing(df)
    print(f"{n}: filled {before} NaN, {df.duplicated().sum()} duplicates removed")

## 3. Outlier Detection (IQR capping)

In [ ]:
def cap_outliers(df, target):
    num = df.select_dtypes(include=np.number).columns.drop(target, errors='ignore')
    for c in num:
        q1, q3 = df[c].quantile([.25,.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
        df[c] = df[c].clip(lo, hi)
    return df

for n,df in loaded.items():
    loaded[n] = cap_outliers(df, DISEASES[n]['target'])
print('outliers capped')

## 4. Feature Engineering

In [ ]:
# Example: add age-group for heart, BMI band for diabetes
if 'heart' in loaded:
    d = loaded['heart']
    if 'age' in d.columns:
        d['age_group'] = pd.cut(d['age'], bins=[0,40,55,70,120],
                                labels=['young','middle','senior','elderly'])
if 'diabetes' in loaded:
    d = loaded['diabetes']
    if 'BMI' in d.columns:
        d['BMI_band'] = pd.cut(d['BMI'], bins=[0,18.5,25,30,100],
                               labels=['under','normal','over','obese'])
print('feature engineering done')

## 5. Encoding (Label + One-Hot)

In [ ]:
def encode(df, target):
    encoders = {}
    # label-encode target if categorical
    if df[target].dtype == 'object':
        le = LabelEncoder(); df[target] = le.fit_transform(df[target].astype(str))
        encoders['__target__'] = le
    # one-hot small-cardinality, label-encode otherwise
    for c in df.select_dtypes(include=['object','category']).columns:
        if c == target: continue
        if df[c].nunique() <= 6:
            df = pd.get_dummies(df, columns=[c], drop_first=True)
        else:
            le = LabelEncoder(); df[c] = le.fit_transform(df[c].astype(str))
            encoders[c] = le
    return df, encoders

encoders_all = {}
for n in list(loaded):
    loaded[n], encoders_all[n] = encode(loaded[n], DISEASES[n]['target'])
    print(n, '->', loaded[n].shape)

## 6. Train/Test Split & StandardScaler

In [ ]:
scalers = {}
splits  = {}
for n,df in loaded.items():
    t = DISEASES[n]['target']
    X = df.drop(columns=[t]); y = df[t]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.2, random_state=42,
                                              stratify=y if y.nunique()<10 else None)
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr); X_te_s = sc.transform(X_te)
    scalers[n] = sc
    splits[n]  = (X_tr_s, X_te_s, y_tr.values, y_te.values, list(X.columns))
    print(f"{n}: train={X_tr.shape} test={X_te.shape}")

## 7. Save Cleaned Dataset

In [ ]:
for n,df in loaded.items():
    df.to_csv(OUT / f'{n}_clean.csv', index=False)
    print(f"saved {OUT / (n+'_clean.csv')}")
# Persist splits + scalers for notebook 2
import joblib
joblib.dump({'splits':splits,'scalers':scalers,'encoders':encoders_all,'diseases':DISEASES},
            OUT/'prepared.joblib')
print('prepared.joblib saved')

---
**Done.** Open `02_model_training_evaluation.ipynb` next.